# R0: clean trial-disjoint Ridge reproduction

Run from top to bottom on a tablet. This notebook first proves data access, then runs a small smoke test, and only then starts the full baseline. Results are saved to Drive. The reported legacy `r=0.8701` is a comparison value, not a pass condition.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DATA_PATH = Path('/content/drive/MyDrive/data_and_scripts/source_data/processed/MM_S1_processed.mat')
OUTPUT_ROOT = Path('/content/drive/MyDrive/pmd1-r0-results')

if not DATA_PATH.is_file():
    print(f'R0_GATE=BLOCKED: {DATA_PATH} not found')
    raise FileNotFoundError('Upload MM_S1_processed.mat or change DATA_PATH above.')
print(f'R0_GATE=DATA_FOUND size_mb={DATA_PATH.stat().st_size / 1_000_000:.1f}')

In [ ]:
import subprocess

REPO = Path('/content/Research-projects')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/youjhun/Research-projects.git', str(REPO)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(REPO / 'reproduction/requirements.txt')], check=True)
print('R0_GATE=ENV_READY')

In [ ]:
subprocess.run(
    ['python', '-m', 'unittest', 'discover', '-s', 'reproduction/tests', '-v'],
    cwd=REPO,
    check=True,
)
print('R0_GATE=CODE_TESTS_PASS')

In [ ]:
SMOKE_OUTPUT = OUTPUT_ROOT / 'smoke'
subprocess.run(
    [
        'python', '-m', 'reproduction.r0_ridge',
        '--data', str(DATA_PATH),
        '--output', str(SMOKE_OUTPUT),
        '--max-trials', '25',
        '--folds', '5',
    ],
    cwd=REPO,
    check=True,
)
print(f'R0_GATE=SMOKE_PASS output={SMOKE_OUTPUT}')

In [ ]:
import json
from pprint import pprint

pprint(json.loads((SMOKE_OUTPUT / 'r0_summary.json').read_text())['result'])

## Full run

Run this cell only after the smoke gate passes. It uses all trials, performs five trial-disjoint folds, hashes the input dataset, and writes the evidence files to Drive.

In [ ]:
FULL_OUTPUT = OUTPUT_ROOT / 'full'
subprocess.run(
    [
        'python', '-m', 'reproduction.r0_ridge',
        '--data', str(DATA_PATH),
        '--output', str(FULL_OUTPUT),
        '--hash-data',
    ],
    cwd=REPO,
    check=True,
)
summary = json.loads((FULL_OUTPUT / 'r0_summary.json').read_text())
print('R0 complete. This value is evidence, not a target to tune toward.')
pprint(summary['result'])
print(f"Evidence saved to {FULL_OUTPUT}")